### Setup

In [1]:
# --- 1. SETUP & INSTALLATION ---

# Import library yang diperlukan
import torch
import os
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from roboflow import Roboflow
from ultralytics import YOLO
import supervision as sv  # [citation:3]
import warnings
warnings.filterwarnings('ignore')

# Menampilkan informasi sistem
print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")

# Mengecek dan mengonfigurasi GPU NVIDIA RTX 2050 (4GB VRAM)
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Device: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    
    # Membersihkan cache CUDA untuk mengoptimalkan memori
    torch.cuda.empty_cache()
    print("CUDA cache cleared successfully")
else:
    device = torch.device("cpu")
    print("GPU not available - using CPU for training")

print("=" * 60)

SYSTEM INFORMATION
Python Version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:16:10) [GCC 11.2.0]
PyTorch Version: 2.9.1+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Device: NVIDIA GeForce RTX 2050
VRAM: 4.0 GB
CUDA cache cleared successfully


### Dataset Configuration

In [2]:
import os
import yaml
from roboflow import Roboflow
from pathlib import Path

# --- 2. DATASET CONFIGURATION (AUTO-FIX) ---
print("Configuring Dataset...")
print("=" * 60)

# 1. Masukkan Path Lokal jika sudah pernah download (Berdasarkan error log Anda sebelumnya)
# Ini path folder dataset yang ada di error log Anda yang lalu
LOCAL_DATASET_PATH = "/media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Dataset-APD-Skripsi-1"

# 2. Cek apakah dataset sudah ada secara lokal?
if os.path.exists(os.path.join(LOCAL_DATASET_PATH, "data.yaml")):
    print(f"✅ Dataset ditemukan secara lokal di: {LOCAL_DATASET_PATH}")
    dataset_location = LOCAL_DATASET_PATH
    data_yaml_path = os.path.join(dataset_location, "data.yaml")

else:
    print("⚠️ Dataset lokal tidak ditemukan/path salah.")
    print("🔄 Mencoba download baru dari Roboflow...")
    
    # KONFIGURASI API (Hanya dipakai jika download baru)
    API_KEY = "2AsxfGTZiBtxpqygwUIv" 
    WORKSPACE = "mrcahyono26"
    PROJECT = "personal-protective-equipment-dtt2i-nrtok"
    VERSION = 2

    try:
        rf = Roboflow(api_key=API_KEY)
        project = rf.workspace(WORKSPACE).project(PROJECT)
        dataset = project.version(VERSION).download("yolov11")
        dataset_location = dataset.location
        data_yaml_path = f"{dataset_location}/data.yaml"
        print(f"✅ Download Sukses ke: {dataset_location}")
        
    except Exception as e:
        print(f"❌ ERROR DOWNLOAD: {e}")
        # Stop script jika download gagal agar tidak lanjut ke training dengan path dummy
        raise RuntimeError("Gagal menyiapkan dataset. Cek koneksi internet atau API Key.")

# --- 3. FIXING YAML PATH (PENTING: Agar tidak error rekursif/missing path) ---
print("🔧 Memperbaiki konfigurasi path di data.yaml...")

try:
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    # Hapus key 'path' yang sering bikin error path ganda
    if 'path' in data_config:
        del data_config['path']

    # Set Absolute Path secara manual agar YOLO tidak bingung
    # Pastikan folder 'train/images', 'valid/images' benar-benar ada
    data_config['train'] = os.path.join(dataset_location, "train", "images")
    data_config['val']   = os.path.join(dataset_location, "valid", "images")
    data_config['test']  = os.path.join(dataset_location, "test", "images")

    # Tulis ulang file yaml dengan konfigurasi yang benar
    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False)
        
    print(f"✅ YAML Fixed: {data_yaml_path}")
    print(f"   Train Path: {data_config['train']}")

except Exception as e:
    print(f"❌ Gagal memperbaiki YAML: {e}")
    raise e

print("=" * 60)

Configuring Dataset...
⚠️ Dataset lokal tidak ditemukan/path salah.
🔄 Mencoba download baru dari Roboflow...
loading Roboflow workspace...
loading Roboflow project...
✅ Download Sukses ke: /media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Personal-Protective-Equipment-2
🔧 Memperbaiki konfigurasi path di data.yaml...
✅ YAML Fixed: /media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Personal-Protective-Equipment-2/data.yaml
   Train Path: /media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Personal-Protective-Equipment-2/train/images


### Model Initialization

In [3]:
# --- 3. MODEL INITIALIZATION ---
# Memuat model YOLOv11 Nano untuk fine-tuning [citation:2][citation:7]
print("Loading YOLOv11 Nano model...")

# Opsi 1: Menggunakan model pre-trained dari Ultralytics (disarankan)
model = YOLO("yolo11n.pt")

# Opsi 2: Membangun model baru dari YAML (opsional)
# model = YOLO("yolo11n.yaml").load("yolo11n.pt")  # YAML + weights

# Verifikasi arsitektur model
print(f"Model loaded: {model.model.__class__.__name__}")
print(f"Model parameters: {sum(p.numel() for p in model.model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.model.parameters() if p.requires_grad):,}")

# Memeriksa jumlah kelas pada model vs dataset
if 'config' in locals():
    model_nc = model.model.yaml['nc'] if hasattr(model.model, 'yaml') else 80
    dataset_nc = config['nc']
    
    if model_nc != dataset_nc:
        print(f"\nWARNING: Model has {model_nc} classes, but dataset has {dataset_nc} classes")
        print("The model will automatically adjust to the dataset classes during training")

Loading YOLOv11 Nano model...
Model loaded: DetectionModel
Model parameters: 2,624,080
Trainable parameters: 0


### Training Configuration

In [4]:
# --- 4. TRAINING CONFIGURATION ---
# Konfigurasi training yang optimal untuk NVIDIA RTX 2050 (4GB VRAM) [citation:2][citation:7]
training_config = {
    "data": data_yaml_path,       # File konfigurasi dataset
    "epochs": 150,                  # Jumlah epoch [citation:2]
    "imgsz": 640,                  # Ukuran input gambar [citation:2][citation:7]
    "batch": 16,                   # Batch size (optimal untuk 4GB VRAM) [citation:2][citation:7]
    "patience": 50,                # Early stopping patience [citation:7]
    "device": device,              # Device (GPU/CPU) [citation:7]
    "workers": 4,                  # Workers untuk data loading (optimalkan untuk laptop) [citation:7]
    "seed": 42,                    # Seed untuk reproducibility [citation:7]
    "deterministic": True,         # Deterministic training [citation:7]
    "amp": True,                   # Automatic Mixed Precision (menghemat VRAM) [citation:7]
    "optimizer": "AdamW",          # Optimizer [citation:7]
    "lr0": 0.001,                  # Initial learning rate [citation:7]
    "lrf": 0.01,                   # Final learning rate factor [citation:7]
    "momentum": 0.937,             # Momentum [citation:7]
    "weight_decay": 0.0005,        # Weight decay [citation:7]
    "warmup_epochs": 3.0,          # Warmup epochs [citation:7]
    "box": 7.5,                    # Box loss gain [citation:7]
    "cls": 0.5,                    # Class loss gain [citation:7]
    "dfl": 1.5,                    # Distribution focal loss gain [citation:7]
    "pretrained": True,            # Gunakan pre-trained weights [citation:7]
    "save": True,                  # Simpan model checkpoint [citation:7]
    "save_period": 10,             # Simpan checkpoint setiap 10 epoch [citation:7]
    "cache": False,                # Cache dataset (False untuk VRAM terbatas) [citation:7]
    "single_cls": False,           # Single class mode (False untuk multi-class) [citation:7]
    "cos_lr": True,                # Cosine learning rate scheduler [citation:7]
    "close_mosaic": 10,            # Nonaktifkan mosaic augmentation di 10 epoch terakhir [citation:7]
    "project": "yolo11_ppe",       # Nama project untuk penyimpanan [citation:7]
    "name": "train_v1",            # Nama training run [citation:7]
    "exist_ok": True,              # Overwrite existing runs [citation:7]
    "augment": True,               # Pastikan augmentasi aktif
    "mosaic": 1.0,                 # Mosaic membantu mendeteksi objek kecil
    "mixup": 0.1,                  # Coba tambahkan mixup sedikit
    "close_mosaic": 15,            # Matikan mosaic 15 epoch terakhir biar stabil
    
}

print("Training Configuration:")
print("=" * 60)
for key, value in training_config.items():
    print(f"{key:20}: {value}")
print("=" * 60)

# Note: Untuk laptop dengan VRAM terbatas:
# - Batch size 16 optimal untuk 4GB VRAM
# - Cache=False untuk menghemat memori
# - Workers=4 untuk menghindari overload CPU laptop
# - amp=True untuk mixed precision training

Training Configuration:
data                : /media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Personal-Protective-Equipment-2/data.yaml
epochs              : 150
imgsz               : 640
batch               : 16
patience            : 50
device              : cuda
workers             : 4
seed                : 42
deterministic       : True
amp                 : True
optimizer           : AdamW
lr0                 : 0.001
lrf                 : 0.01
momentum            : 0.937
weight_decay        : 0.0005
warmup_epochs       : 3.0
box                 : 7.5
cls                 : 0.5
dfl                 : 1.5
pretrained          : True
save                : True
save_period         : 10
cache               : False
single_cls          : False
cos_lr              : True
close_mosaic        : 15
project             : yolo11_ppe
name                : train_v1
exist_ok            : True
augment             : True
mosaic              : 1.0
mixup               : 0.1


### Model Training

In [5]:
# --- 5. MODEL TRAINING ---
print("Starting YOLOv11 Nano training on PPE dataset...")
print("=" * 60)

# Memulai training [citation:2]
try:
    # Training dengan konfigurasi yang telah ditentukan
    results = model.train(**training_config)
    
    print("Training completed successfully!")
    
    # Menampilkan hasil training terbaik
    if hasattr(results, 'best_fitness'):
        print(f"Best mAP50-95: {results.best_fitness:.4f}")
    
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("CUDA Out of Memory Error!")
        print("Reducing batch size to 8...")
        
        # Mengurangi batch size untuk mengatasi OOM error
        training_config["batch"] = 8
        print("Retraining with reduced batch size...")
        results = model.train(**training_config)
    else:
        raise e

print("=" * 60)

Starting YOLOv11 Nano training on PPE dataset...


engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Personal-Protective-Equipment-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=50, perspective=0.0, plots=True, pose=12.

### Export Model

In [6]:
from ultralytics import YOLO
import pandas as pd
import json

# 1. Tentukan path model
model_path = 'yolo11_ppe/train_v1/weights/best.pt'

# 2. Load model ke memori global
print(f"Memuat model dari: {model_path}")
model = YOLO(model_path)

# 3. Jalankan Validasi Ulang untuk mendapatkan 'metrics'
# Ini penting agar kita bisa mengisi nilai mAP, Precision, dll.
print("Menjalankan validasi untuk mendapatkan metrik performa...")
metrics = model.val() 

# 4. Fungsi Export (Sama seperti sebelumnya)
def export_model(model_instance, export_formats=['onnx', 'engine', 'torchscript']):
    """
    Export model ke format yang berbeda untuk deployment
    """
    exported_path = ""
    for fmt in export_formats:
        try:
            print(f"\nExporting ke {fmt}...")
            
            if fmt == 'onnx':
                exported_path = model_instance.export(
                    format='onnx',
                    imgsz=640,
                    opset=12,
                    simplify=True,
                    dynamic=True,
                    half=False
                )
            # (Format lain bisa ditambahkan di sini jika perlu)
            
            print(f"Model {fmt} disimpan: {exported_path}")
            
        except Exception as e:
            print(f"Error exporting ke {fmt}: {e}")
    
    return exported_path

# 5. Jalankan Export
print("\nExporting model untuk deployment...")
exported_model_path = export_model(model, ['onnx'])

# 6. Simpan Metadata (Sekarang aman karena 'metrics' dan 'model' sudah ada)
model_info = {
    'model_name': 'yolov11n_ppe_detector',
    'version': '1.0',
    'classes': model.names,
    'input_size': 640,
    # 'training_epochs': 50, # Opsional: hardcode atau ambil dari log jika ada
    'mAP50': float(metrics.box.map50),
    'mAP50_95': float(metrics.box.map),
    'precision': float(metrics.box.p.mean()), # Ambil rata-rata precision
    'recall': float(metrics.box.r.mean()),    # Ambil rata-rata recall
    'export_formats': ['pt', 'onnx'],
    'export_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open('model_metadata.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("\n=== EKSPOR SELESAI ===")
print(f"1. Model PyTorch: {model_path}")
print(f"2. Model ONNX: {exported_model_path}")
print(f"3. Metadata: model_metadata.json")

Memuat model dari: yolo11_ppe/train_v1/weights/best.pt
Menjalankan validasi untuk mendapatkan metrik performa...
Ultralytics 8.3.241 🚀 Python-3.13.9 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 2050, 3771MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.1 ms, read: 91.8±61.6 MB/s, size: 33.2 KB)
val: Scanning /media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Personal-Protective-Equipment-2/valid/labels.cache... 265 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 265/265 247.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 4.3it/s 4.0s0.2s
                   all        265        841      0.686      0.752      0.746      0.417
                Gloves         67        144      0.712      0.924      0.861      0.491
                Helmet        119        229       0.81      0.373      0.687      0.369
             No

### validation and Metrics Visualization

In [7]:
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# --- 6. VALIDATION & METRICS VISUALIZATION ---
print("Validating trained model...")

# Path model (sesuaikan jika perlu)
# Jika variabel training_config sudah hilang dari memori, hardcode path-nya:
# best_model_path = Path("yolo11_ppe/train_v1/weights/best.pt") 
best_model_path = Path(training_config["project"]) / training_config["name"] / "weights" / "best.pt"
print(f"Best model path: {best_model_path}")

if best_model_path.exists():
    # Load Model
    best_model = YOLO(str(best_model_path))
    
    # Jalankan Validasi
    metrics = best_model.val(
        data=data_yaml_path,
        split="val",
        imgsz=640,
        batch=8,
        device=device,
        plots=True
    )
    
    # --- MENAMPILKAN METRIK UTAMA (DIPERBAIKI) ---
    print("\nVALIDATION METRICS:")
    print("=" * 60)
    
    # FIX: Menggunakan mean() untuk menangani array multi-class
    precision = metrics.box.p.mean() if hasattr(metrics.box.p, 'mean') else metrics.box.p
    recall = metrics.box.r.mean() if hasattr(metrics.box.r, 'mean') else metrics.box.r
    
    print(f"Precision (P): {precision:.4f}")
    print(f"Recall (R):    {recall:.4f}")
    print(f"mAP@50:        {metrics.box.map50:.4f}")
    print(f"mAP@50-95:     {metrics.box.map:.4f}")
    
    # Metrik Per Kelas
    print("\nPER-CLASS METRICS:")
    # Mengambil nama kelas dari model langsung biar akurat
    class_names = best_model.names
    for i, class_name in class_names.items():
        # Maps array index i to the AP50 value for that class
        # Hati-hati: metrics.box.maps shape adalah (num_classes, num_thresholds)
        # AP50 biasanya ada di index 0 dari threshold (tergantung implementasi ultralytics)
        # Cara paling aman ambil dari maps index 0 (AP@50)
        ap50 = metrics.box.maps[i] # map50-95 specific for class i
        print(f"{class_name:20}: mAP50-95 = {ap50:.4f}")
    
    print("=" * 60)
    
    # --- VISUALISASI PLOT BAWAAN YOLO (SUDAH LENGKAP) ---
    results_dir = Path(training_config["project"]) / training_config["name"]
    print(f"\nVisualizing plots from: {results_dir}")
    
    plots_to_show = [
        "confusion_matrix.png",
        "results.png",
        "F1_curve.png",
        "PR_curve.png"
    ]
    
    # Setup Grid Plot
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, plot_name in enumerate(plots_to_show):
        plot_path = results_dir / plot_name
        if plot_path.exists():
            img = plt.imread(str(plot_path))
            axes[idx].imshow(img)
            axes[idx].axis('off')
            axes[idx].set_title(plot_name)
        else:
            axes[idx].text(0.5, 0.5, f"{plot_name}\nNot Found", ha='center')
    
    plt.tight_layout()
    plt.show()

else:
    print(f"❌ Model file not found at {best_model_path}")

Validating trained model...
Best model path: yolo11_ppe/train_v1/weights/best.pt
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 112.0±29.0 MB/s, size: 37.9 KB)
val: Scanning /media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Personal-Protective-Equipment-2/valid/labels.cache... 265 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 265/265 297.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 9.1it/s 3.7s<0.2s
                   all        265        841      0.687      0.751      0.746      0.417
                Gloves         67        144      0.712      0.924      0.861      0.491
                Helmet        119        229       0.81      0.373      0.687      0.369
             No-Helmet         43         81       0.61      0.605       0.64      0.302
             No-gloves         93        159      0.

<Figure size 1500x1000 with 4 Axes>

### Inference Test

In [8]:
import matplotlib.pyplot as plt
from pathlib import Path

# --- 7. INFERENCE TEST (FIXED) ---
print("Running inference test on sample image...")

# Mencari gambar test
# Pastikan variabel 'dataset' masih ada. Jika error 'dataset not defined', 
# ganti 'dataset.location' dengan path manual string.
test_images_dir = Path(dataset.location) / "test/images"
test_images = list(test_images_dir.glob("*.jpg"))

if test_images:
    # Mengambil gambar test pertama
    sample_image_path = test_images[0]
    print(f"Test image: {sample_image_path.name}")
    
    # Menjalankan inference
    results = best_model.predict(
        source=str(sample_image_path),
        conf=0.25,
        iou=0.45,
        imgsz=640,
        device=device,
        save=False
    )
    
    # Visualisasi
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Gambar asli
    original_img = plt.imread(str(sample_image_path))
    axes[0].imshow(original_img)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Gambar hasil prediksi
    result_img = results[0].plot()
    axes[1].imshow(result_img)
    
    # Statistik
    detections = results[0].boxes
    num_detections = len(detections) if detections is not None else 0
    
    axes[1].set_title(f"Predictions (Detections: {num_detections})")
    axes[1].axis('off')
    
    plt.suptitle("YOLOv11 Nano Inference Test - PPE Detection", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Detail Deteksi
    if num_detections > 0:
        print(f"\nDetected {num_detections} object(s):")
        print("-" * 50)
        for i, box in enumerate(detections):
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            
            # --- PERBAIKAN DI SINI ---
            # Mengambil nama kelas langsung dari model, bukan dari variabel 'config' luar
            class_name = best_model.names[class_id] 
            
            print(f"{i+1}. {class_name}: {confidence:.2%}")
    
    # Waktu Inference
    # speed adalah dictionary, kita ambil rata-rata totalnya
    inference_time = results[0].speed.get('inference', 0)
    print(f"\nInference time: {inference_time:.1f}ms")
    
else:
    print("No test images found. Trying training images...")
    # Fallback ke folder train jika test kosong
    train_images = list((Path(dataset.location) / "train/images").glob("*.jpg"))
    if train_images:
        print(f"Found image in train: {train_images[0].name}")
        # (Anda bisa copy paste logika inference di atas ke sini jika perlu)

print("=" * 60)

Running inference test on sample image...
Test image: IMG_4866_21-31sec_jpg.rf.8b7050db66c4d7701481286634e08624.jpg

image 1/1 /media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Personal-Protective-Equipment-2/test/images/IMG_4866_21-31sec_jpg.rf.8b7050db66c4d7701481286634e08624.jpg: 640x640 2 No-glovess, 7.2ms
Speed: 3.3ms preprocess, 7.2ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 1500x600 with 2 Axes>


Detected 2 object(s):
--------------------------------------------------
1. No-gloves: 85.92%
2. No-gloves: 73.02%

Inference time: 7.2ms


### Model Export

In [9]:
# --- 8. MODEL EXPORT ---
# Mengekspor model untuk deployment [citation:4]
print("Exporting model for deployment...")

export_formats = {
    "ONNX": "onnx",          # Untuk inferensi CPU (3x speedup) [citation:4]
    "TensorRT": "engine",    # Untuk GPU NVIDIA (5x speedup) [citation:4]
    "TorchScript": "torchscript",  # Untuk PyTorch mobile
}

# Mengekspor ke berbagai format
exported_models = {}

for format_name, format_arg in export_formats.items():
    try:
        print(f"\nExporting to {format_name} format...")
        
        # Konfigurasi ekspor berdasarkan format [citation:4]
        export_kwargs = {
            "format": format_arg,
            "imgsz": 640,
            "device": device if device.type == "cuda" else "cpu",
        }
        
        # Konfigurasi khusus untuk beberapa format
        if format_arg == "onnx":
            export_kwargs["simplify"] = True
            export_kwargs["dynamic"] = False  # Fixed size untuk performa lebih baik
            
        elif format_arg == "engine":  # TensorRT
            export_kwargs["half"] = True  # FP16 untuk kecepatan
            export_kwargs["workspace"] = 2  # 2GB workspace untuk RTX 2050
            
        elif format_arg == "torchscript":
            export_kwargs["optimize"] = True
            
        # Mengekspor model
        export_path = best_model.export(**export_kwargs)
        exported_models[format_name] = export_path
        
        print(f"✓ {format_name} model exported: {Path(export_path).name}")
        
        # Verifikasi model yang diekspor
        if Path(export_path).exists():
            file_size_mb = Path(export_path).stat().st_size / (1024 * 1024)
            print(f"  File size: {file_size_mb:.1f} MB")
        
    except Exception as e:
        print(f"✗ Failed to export to {format_name}: {str(e)}")

print("\n" + "=" * 60)
print("EXPORT SUMMARY:")
print("=" * 60)
for format_name, path in exported_models.items():
    print(f"{format_name:12}: {path}")

# Simpan best.pt ke lokasi yang mudah diakses
final_model_path = Path("best_ppe_yolo11n.pt")
if best_model_path.exists():
    import shutil
    shutil.copy2(best_model_path, final_model_path)
    print(f"\nBest model copied to: {final_model_path}")

print("\n" + "=" * 60)
print("DEPLOYMENT READY!")
print("=" * 60)
print("\nModel can be deployed using:")
print("1. Ultralytics Python API:")
print(f"   model = YOLO('{final_model_path}')")
print("   results = model.predict(source='image.jpg')")
print("\n2. Roboflow Deployment (for API endpoints):")
print("   Visit: https://roboflow.com/deploy")
print("\n3. TensorRT for GPU acceleration:")
print("   Use the exported .engine file with TensorRT runtime")
print("\n4. ONNX Runtime for CPU inference:")
print("   Use the exported .onnx file with ONNX Runtime")

print("\n" + "=" * 60)
print("TRAINING PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 60)

Exporting model for deployment...

Exporting to ONNX format...

PyTorch: starting from 'yolo11_ppe/train_v1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (5.2 MB)

ONNX: starting export with onnx 1.20.0 opset 20...
ONNX: slimming with onnxslim 0.1.80...
ONNX: export success ✅ 1.1s, saved as 'yolo11_ppe/train_v1/weights/best.onnx' (10.1 MB)

Export complete (1.2s)
Results saved to /media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/yolo11_ppe/train_v1/weights
Predict:         yolo predict task=detect model=yolo11_ppe/train_v1/weights/best.onnx imgsz=640  
Validate:        yolo val task=detect model=yolo11_ppe/train_v1/weights/best.onnx imgsz=640 data=/media/mrcahyono/Data/1. Kuliah/Semester 6/proyek_informatika/app/Personal-Protective-Equipment-2/data.yaml  
Visualize:       https://netron.app
✓ ONNX model exported: best.onnx
  File size: 10.1 MB

Exporting to TensorRT format...

PyTorch: starting from 'yolo11_ppe/train_v1/weight